# Practice Lab: BERT and Encoder Models (Lesson 4.3)

Module 4 · 6 exercises · ~90-120 min
**Needs:** transformers, datasets, scikit-learn


# Lesson 4.3: BERT and Encoder Models

6 exercises: 2 Easy + 2 Medium + 2 Hard
**Needs:** `pip install transformers datasets scikit-learn`


In [ ]:
!pip install transformers datasets scikit-learn -q


---
## Exercise 1 (Easy): Explore MLM + Bias


In [ ]:
# YOUR CODE
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
    "The capital of India is [MASK].",
    "Python is a popular [MASK] language.",
    "I ordered butter [MASK] for dinner.",
    "The [MASK] chased the mouse.",
    "This man works as a [MASK].",
    "This woman works as a [MASK].",
]

for sent in sentences:
    results = unmasker(sent)[:3]
    print(f"\n  {sent}")
    for r in results:
        print(f"    -> '{r['token_str']}' ({r['score']:.0%})")

# TODO: analyze man vs woman bias


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
    "The capital of India is [MASK].",
    "Python is a popular [MASK] language.",
    "I ordered butter [MASK] for dinner.",
    "After winning, they felt [MASK].",
    "The [MASK] chased the mouse.",
    "This man works as a [MASK].",
    "This woman works as a [MASK].",
    "The [MASK] nurse helped the patient.",
    "The [MASK] engineer designed the bridge.",
    "People from [MASK] are very friendly.",
]

for sent in sentences:
    results = unmasker(sent)[:3]
    print(f"\n  {sent}")
    for r in results:
        print(f"    -> '{r['token_str']}' ({r['score']:.0%})")

# Bias analysis
print("\n=== BIAS ANALYSIS ===")
for gender in ["man", "woman"]:
    sent = f"This {gender} works as a [MASK]."
    results = unmasker(sent)[:5]
    print(f"\n  '{gender}' top jobs:")
    for r in results:
        print(f"    {r['token_str']:15s} {r['score']:.0%}")

print("\nBiases found:")
print("  - Gender stereotypes in job predictions")
print("  - 'man' -> lawyer/doctor, 'woman' -> nurse/maid")
print("  - Production: audit, debias, add guardrails")


</details>


---
## Exercise 2 (Easy): Compare Models


In [ ]:
# YOUR CODE
from transformers import pipeline
import time

models = [
    ("BERT", "textattack/bert-base-uncased-SST-2"),
    ("DistilBERT",
     "distilbert-base-uncased-finetuned-sst-2-english"),
    ("RoBERTa", "textattack/roberta-base-SST-2"),
]

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie ever. Complete waste of time.",
    "It was okay. Nothing special.",
    "Visuals stunning but story was weak.",
    "A masterpiece. Will watch again!",
]

# TODO: time each model, print comparison table


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline
import time

models = [
    ("BERT", "textattack/bert-base-uncased-SST-2"),
    ("DistilBERT",
     "distilbert-base-uncased-finetuned-sst-2-english"),
    ("RoBERTa", "textattack/roberta-base-SST-2"),
]

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie ever. Complete waste of time.",
    "It was okay. Nothing special.",
    "Visuals stunning but story was weak.",
    "A masterpiece. Will watch again!",
]

results = {}
for name, model_id in models:
    pipe = pipeline("sentiment-analysis", model=model_id)
    t0 = time.time()
    preds = pipe(reviews)
    ms = (time.time() - t0) / len(reviews) * 1000
    results[name] = {"ms": ms, "preds": preds}
    print(f"  {name:12s}: {ms:.0f}ms/review")

print("\nPer-review comparison:")
for i, rev in enumerate(reviews):
    print(f"\n  '{rev[:40]}...'")
    for name in results:
        p = results[name]["preds"][i]
        print(f"    {name:12s}: "
              f"{p['label']:>8s} ({p['score']:.0%})")

print("\nBest production choice: DistilBERT")
print("  40% smaller, 60% faster, 95% accuracy")


</details>


---
## Exercise 4 (Medium): [CLS] Vector Explorer


In [ ]:
# YOUR CODE
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

sentences = [
    "The team won the championship game.",
    "He scored a hat trick in football.",
    "Cricket is India's favorite sport.",
    "Biryani is a delicious rice dish.",
    "I ordered butter chicken for dinner.",
    "Fresh dosa with coconut chutney.",
    "Python is a popular programming language.",
    "Machine learning predicts outcomes.",
    "GPUs accelerate neural network training.",
    "The acting in this film was superb.",
    "I cried during the emotional scenes.",
    "Best movie I have seen this year.",
]

def get_cls(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[0, 0].numpy()

# TODO: extract [CLS] vectors, compute similarity
# TODO: plot heatmap


<details><summary>Solution</summary>


In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

sentences = [
    "The team won the championship game.",
    "He scored a hat trick in football.",
    "Cricket is India's favorite sport.",
    "Biryani is a delicious rice dish.",
    "I ordered butter chicken for dinner.",
    "Fresh dosa with coconut chutney.",
    "Python is a popular programming language.",
    "Machine learning predicts outcomes.",
    "GPUs accelerate neural network training.",
    "The acting in this film was superb.",
    "I cried during the emotional scenes.",
    "Best movie I have seen this year.",
]

topics = ["Sport"]*3 + ["Food"]*3 + ["Tech"]*3 + ["Movie"]*3

def get_cls(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[0, 0].numpy()

vecs = np.array([get_cls(s) for s in sentences])
sim = cosine_similarity(vecs)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='RdYlGn', vmin=0.85, vmax=1.0)
for i, t in enumerate(topics):
    ax.text(-0.5, i, t, ha='right', fontsize=7, va='center')
plt.colorbar(im)
ax.set_title('[CLS] Cosine Similarity')
plt.tight_layout()
plt.savefig('cls_similarity.png', dpi=150)
plt.show()

# Within vs between topic
topic_ids = [0]*3 + [1]*3 + [2]*3 + [3]*3
within, between = [], []
for i in range(12):
    for j in range(i+1, 12):
        if topic_ids[i] == topic_ids[j]:
            within.append(sim[i,j])
        else:
            between.append(sim[i,j])
print(f"Avg within-topic:  {np.mean(within):.3f}")
print(f"Avg between-topic: {np.mean(between):.3f}")


</details>


---
## Exercise 3 (Medium): Hindi Sentiment with IndicBERT
See practice-lab-4_3.html for full details.


In [ ]:
# YOUR CODE
from transformers import pipeline

# Load both models for fill-mask
indic = pipeline("fill-mask",
    model="ai4bharat/indic-bert")
mbert = pipeline("fill-mask",
    model="bert-base-multilingual-cased")

hindi_sentences = [
    # "This movie was very [MASK]."
    "यह फिल्म बहुत [MASK] थी।",
    # "The [MASK] was delicious."
    "[MASK] बहुत स्वादिष्ट था।",
    # "India is a [MASK] country."
    "भारत एक [MASK] देश है।",
]

for sent in hindi_sentences:
    print(f"\n  Sentence: {sent}")
    print("  IndicBERT:", indic(sent)[0]['token_str'])
    print("  mBERT:    ", mbert(sent)[0]['token_str'])

# TODO: test on 10 Hindi sentences
# TODO: score naturalness of completions


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline

indic = pipeline("fill-mask",
    model="ai4bharat/indic-bert")
mbert = pipeline("fill-mask",
    model="bert-base-multilingual-cased")

hindi_sentences = [
    "यह फिल्म बहुत [MASK] थी।",
    "[MASK] बहुत स्वादिष्ट था।",
    "भारत एक [MASK] देश है।",
]

for sent in hindi_sentences:
    print(f"\n  Sentence: {sent}")
    print("  IndicBERT:", indic(sent)[0]['token_str'])
    print("  mBERT:    ", mbert(sent)[0]['token_str'])

# Score each model's completions
scores = {"indic": 0, "mbert": 0}
for sent in hindi_sentences:
    i_result = indic(sent)[0]
    m_result = mbert(sent)[0]
    # Higher confidence = more natural
    if i_result['score'] > m_result['score']:
        scores["indic"] += 1
    else:
        scores["mbert"] += 1

print(f"\nIndicBERT wins: {scores['indic']}")
print(f"mBERT wins:     {scores['mbert']}")


</details>


---
## Exercise 5 (Hard): Custom News Classifier (ag_news)
See practice-lab-4_3.html for full details.


In [ ]:
# YOUR CODE
# pip install transformers datasets scikit-learn
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from datasets import load_dataset
import numpy as np

dataset = load_dataset("ag_news")
train = dataset["train"].shuffle(42).select(range(2000))
test  = dataset["test"].shuffle(42).select(range(500))

labels = ["World", "Sports", "Business", "Sci/Tech"]
print(f"Example: '{train[0]['text'][:60]}...'")
print(f"  Label: {labels[train[0]['label']]}")

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased")

def tokenize(examples):
    return tokenizer(examples["text"],
        truncation=True, padding="max_length",
        max_length=128)

train_tok = train.map(tokenize, batched=True)
test_tok  = test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4)

# TODO: set up TrainingArguments, Trainer
# TODO: train, evaluate
# TODO: print confusion matrix and F1 scores


<details><summary>Solution</summary>


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from datasets import load_dataset
from sklearn.metrics import (
    classification_report, confusion_matrix)
import numpy as np

dataset = load_dataset("ag_news")
train = dataset["train"].shuffle(42).select(range(2000))
test  = dataset["test"].shuffle(42).select(range(500))

labels = ["World", "Sports", "Business", "Sci/Tech"]

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased")

def tokenize(examples):
    return tokenizer(examples["text"],
        truncation=True, padding="max_length",
        max_length=128)

train_tok = train.map(tokenize, batched=True)
test_tok  = test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    acc = (preds == eval_pred.label_ids).mean()
    return {"accuracy": acc}

args = TrainingArguments(
    output_dir="./news_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    logging_steps=100,
    save_strategy="no",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)
trainer.train()

# Detailed metrics
preds = trainer.predict(test_tok)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = np.array(test_tok["label"])
print(classification_report(y_true, y_pred,
                            target_names=labels))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


</details>


---
## Exercise 6 (Challenge): Named Entity Recognition
See practice-lab-4_3.html for full details.


In [ ]:
# YOUR CODE
from transformers import pipeline

# Quick test: pre-trained NER
ner = pipeline("ner", model="dslim/bert-base-NER",
               aggregation_strategy="simple")

tests = [
    "Sundar Pichai works at Google in Mountain View.",
    "Narendra Modi visited the Taj Mahal in Agra.",
    "Apple released the iPhone in Cupertino.",
]

for text in tests:
    entities = ner(text)
    print(f"\n  '{text}'")
    for e in entities:
        print(f"    {e['entity_group']:>4s}: "
              f"'{e['word']}' ({e['score']:.0%})")

# TODO: Load conll2003 dataset
# TODO: Fine-tune BERT for token classification
# TODO: Evaluate per-entity F1 scores


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline, AutoTokenizer
from datasets import load_dataset

# Quick test: pre-trained NER
ner = pipeline("ner", model="dslim/bert-base-NER",
               aggregation_strategy="simple")

tests = [
    "Sundar Pichai works at Google in Mountain View.",
    "Narendra Modi visited the Taj Mahal in Agra.",
    "Apple released the iPhone in Cupertino.",
]

for text in tests:
    entities = ner(text)
    print(f"\n  '{text}'")
    for e in entities:
        print(f"    {e['entity_group']:>4s}: "
              f"'{e['word']}' ({e['score']:.0%})")

# Fine-tune BERT for token classification
dataset = load_dataset("conll2003",
    trust_remote_code=True)

# NER labels
labels = dataset["train"].features["ner_tags"].feature.names
# ['O','B-PER','I-PER','B-ORG','I-ORG',
#  'B-LOC','I-LOC','B-MISC','I-MISC']

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True, padding="max_length",
        max_length=128, is_split_into_words=True)
    all_labels = []
    for i, labels_list in enumerate(
        examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = [-100 if wid is None
                     else labels_list[wid]
                     for wid in word_ids]
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized


</details>
